# Project1 — Môi trường Google Colab
Chạy lần lượt các ô từ trên xuống. Mặc định chỉ thiết lập và kiểm tra môi trường, chưa huấn luyện hoặc tải mô hình.
Mô hình cơ sở PhoBERT CLS + LightGBM chạy trên CPU. Các thư viện LoRA tùy chọn dùng để chuẩn bị cho những thử nghiệm sau này.
Xem hướng dẫn chi tiết tại `docs/COLAB_SETUP.md`. Không cần cài MCP để sử dụng notebook.


## 1. Cấu hình và kết nối Google Drive
Nếu kho mã nguồn ở chế độ riêng tư hoặc có thay đổi trên máy chưa được đẩy lên GitHub, hãy sao chép mã nguồn lên Drive và chọn `SOURCE_MODE = "drive"`. Không sao chép thông tin đăng nhập, khóa bí mật hoặc thư mục môi trường ảo.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys
import shutil

SOURCE_MODE = "github"  # "github" or "drive"
REPO_URL = "https://github.com/Drista4real/Project1.git"
REPO_REF = "develop"  # Use a commit SHA for reproducibility
DRIVE_SOURCE = Path("/content/drive/MyDrive/Project1/source")
INSTALL_LORA = False
RUN_CLASSIFIER = False
RUN_FORECASTER = False
DOWNLOAD_ZIP = False
DATASET_DIR = None  # Example: "/content/drive/MyDrive/Project1/datasets/bundle_v2"
FORECAST_CSV = None  # None uses data/processed/balance_fixed.csv

from google.colab import drive

drive.mount("/content/drive")
WORK_ROOT = Path("/content/project1_colab")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DRIVE = Path("/content/drive/MyDrive/Project1")
from uuid import uuid4
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = PROJECT_DRIVE / "colab_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CACHE_DIR = PROJECT_DRIVE / "model_cache/phobert"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Run artifacts:", RUN_DIR)


## 2. Nạp mã nguồn
Mỗi lần chạy sử dụng một bản sao mã nguồn mới trên ổ đĩa tạm của Colab. Nhánh hoặc phiên bản GitHub được chọn phải chứa các tệp khai báo thư viện mới; nếu chưa có, hãy dùng bản mã nguồn đã cập nhật trên Drive.


In [ ]:
REPO_DIR = WORK_ROOT / RUN_ID
if SOURCE_MODE == "github":
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_REF], check=True)
elif SOURCE_MODE == "drive":
    if not (DRIVE_SOURCE / "backend/scripts/train_classifier.py").is_file():
        raise FileNotFoundError(f"Project source not found at {DRIVE_SOURCE}")
    shutil.copytree(DRIVE_SOURCE, REPO_DIR, ignore=shutil.ignore_patterns(
        ".git", ".venv*", "venv", "node_modules", "__pycache__", ".env", ".env.*",
        "weights", "frontend", ".dart_tool", "build"))
else:
    raise ValueError("SOURCE_MODE must be github or drive")
for name in ("backend/requirements-colab.txt", "backend/scripts/prepare_classifier_data.py"):
    if not (REPO_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {name}; use updated project source")
if SOURCE_MODE == "github":
    commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
else:
    commit = "Drive source snapshot; no verified Git commit"
(RUN_DIR / "source-commit.txt").write_text(commit + "\n", encoding="utf-8")
print("Source:", REPO_DIR, "\nRevision:", commit)


## 3. Cài đặt thư viện học máy
Quá trình cài đặt có thể mất vài phút. Không cần cài các thư viện phục vụ API hoặc cơ sở dữ liệu. Các bước học máy tiếp theo chạy trong tiến trình Python mới để sử dụng đúng phiên bản thư viện vừa cài.


In [ ]:
requirements = "requirements-colab-lora.txt" if INSTALL_LORA else "requirements-colab.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "backend" / requirements)], check=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "pip-freeze.txt").write_text(freeze, encoding="utf-8")


## 4. Kiểm tra môi trường
Bước này không tải mô hình. Nếu có GPU, chương trình sẽ kiểm tra bằng một phép toán CUDA nhỏ. Mô hình cơ sở hiện tại vẫn chạy trên CPU.


In [ ]:
def run_python(source, *args):
    return subprocess.run([sys.executable, "-c", source, *map(str, args)],
                          cwd=REPO_DIR, check=True, env={**os.environ, "PYTHONUTF8": "1"})

run_python("""
import importlib.metadata as meta
import json, platform, sys
from pathlib import Path
import numpy, torch, lightgbm, sklearn, transformers
from underthesea import word_tokenize
assert word_tokenize('\u0103n tr\u01b0a', format='text')
cuda = torch.cuda.is_available()
if cuda:
    assert (torch.ones(2, device='cuda') + 1).sum().item() == 4
packages = ['numpy', 'torch', 'lightgbm', 'scikit-learn', 'transformers', 'underthesea']
if sys.argv[2] == 'True':
    import peft, accelerate, datasets
    packages += ['peft', 'accelerate', 'datasets']
report = {'python': sys.version, 'platform': platform.platform(),
          'cuda_available': cuda, 'torch_cuda': torch.version.cuda,
          'gpu': torch.cuda.get_device_name(0) if cuda else None,
          'versions': {name: meta.version(name) for name in packages}}
Path(sys.argv[1]).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(report, ensure_ascii=False, indent=2))
""", RUN_DIR / "environment.json", INSTALL_LORA)


## 5. Kiểm tra dữ liệu
Trước khi huấn luyện, kiểm tra mã băm và các tập huấn luyện, xác thực, kiểm thử đã được cố định trong gói dữ liệu phân loại. Bạn vẫn có thể hoàn tất thiết lập môi trường khi chưa nhận được dữ liệu phân loại. Dữ liệu số dư được kiểm tra mà không in nội dung giao dịch.


In [ ]:
CLASSIFIER_READY = False
if DATASET_DIR is None:
    print("No classifier bundle yet. Keep RUN_CLASSIFIER=False until data handoff.")
else:
    DATASET_DIR = Path(DATASET_DIR)
    subprocess.run([sys.executable, "backend/scripts/prepare_classifier_data.py",
                    "--verify", str(DATASET_DIR)], cwd=REPO_DIR, check=True)
    CLASSIFIER_READY = True
FORECAST_PATH = Path(FORECAST_CSV) if FORECAST_CSV else REPO_DIR / "data/processed/balance_fixed.csv"
run_python("""
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'backend'))
from app.infrastructure.ai.forecaster.features import load_daily_cashflows
rows = load_daily_cashflows(Path(sys.argv[1]))
if len(rows) < 10:
    raise ValueError('Forecaster needs at least 10 consecutive days')
print('Valid fixed-income CSV:', len(rows), 'days')
""", FORECAST_PATH)


## 6. Huấn luyện mô hình cơ sở (tùy chọn)
Bật các biến điều khiển huấn luyện trong ô cấu hình rồi chạy lại các ô từ đầu, hoặc thay đổi các biến ngay tại đây. Mô hình phân loại sẽ tải PhoBERT trong lần sử dụng đầu tiên. Bước này chưa triển khai huấn luyện LoRA. Nhật ký và các tệp mô hình sau huấn luyện được lưu trên Drive.


In [ ]:
def train_logged(script, arguments, log_name):
    command = [sys.executable, "-u", str(REPO_DIR / script), *map(str, arguments)]
    with (RUN_DIR / log_name).open("w", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True,
                              encoding="utf-8", errors="replace",
                              env={**os.environ, "PYTHONUTF8": "1"}) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"Training failed (exit={status}); see {log_name}")

if RUN_CLASSIFIER:
    if not CLASSIFIER_READY:
        raise ValueError("A verified DATASET_DIR is required before classifier training")
    # The training script verifies the bundle again before using it.
    train_logged("backend/scripts/train_classifier.py", [
        "--dataset-dir", DATASET_DIR, "--output-dir", RUN_DIR / "classifier",
        "--cache-dir", CACHE_DIR, "--batch-size", 16], "classifier.log")
if RUN_FORECASTER:
    train_logged("backend/scripts/train_forecaster.py", [
        "--data", FORECAST_PATH, "--output-dir", RUN_DIR / "forecaster"], "forecaster.log")
if not RUN_CLASSIFIER and not RUN_FORECASTER:
    print("Environment prepared; training has not been requested.")


## 7. Xuất kết quả của lần chạy
Tệp ZIP chứa thông tin môi trường, nhật ký và các tệp mô hình đã lưu nếu có. Bộ nhớ đệm PhoBERT được lưu riêng trên Drive; cần sao chép thêm bộ nhớ đệm này khi triển khai mô hình cơ sở.


In [ ]:
archive = shutil.make_archive(str(WORK_ROOT / RUN_ID), "zip", root_dir=RUN_DIR)
saved_archive = PROJECT_DRIVE / "colab_runs" / (RUN_ID + ".zip")
shutil.copy2(archive, saved_archive)
print("Saved:", saved_archive)
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)
